**A. Feature Engineering**

In [2]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import StringIndexer, VectorAssembler

spark = SparkSession.builder \
    .appName("Tugas11_Dany_Akhdan_Imaduddin") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

df_tugas = spark.read.csv("tugas11_penjualan.csv", header=True, inferSchema=True)
print("Total baris dataset tugas:", df_tugas.count())

# Fit StringIndexer dan simpan modelnya agar bisa dipakai ulang di Bagian E
indexer_menu = StringIndexer(inputCol="nama_menu", outputCol="nama_menu_idx").fit(df_tugas)
indexer_cabang = StringIndexer(inputCol="cabang", outputCol="cabang_idx").fit(df_tugas)
indexer_waktu = StringIndexer(inputCol="waktu_pesan", outputCol="waktu_pesan_idx").fit(df_tugas)

df_tugas_idx = indexer_menu.transform(df_tugas)
df_tugas_idx = indexer_cabang.transform(df_tugas_idx)
df_tugas_idx = indexer_waktu.transform(df_tugas_idx)

assembler_tugas = VectorAssembler(
    inputCols=["qty", "harga_satuan", "nama_menu_idx", "cabang_idx", "waktu_pesan_idx"],
    outputCol="features"
)
df_tugas_final = assembler_tugas.transform(df_tugas_idx)

print("Hasil Feature Engineering (5 baris pertama):")
df_tugas_final.select("nama_menu", "cabang", "waktu_pesan", "qty", "harga_satuan", "features", "total_penjualan").show(5, truncate=False)

Total baris dataset tugas: 2500
Hasil Feature Engineering (5 baris pertama):
+--------------+----------+-----------+---+------------+-------------------------+------------------+
|nama_menu     |cabang    |waktu_pesan|qty|harga_satuan|features                 |total_penjualan   |
+--------------+----------+-----------+---+------------+-------------------------+------------------+
|Kentang Goreng|Semarang  |Malam      |1  |30000       |[1.0,30000.0,5.0,2.0,1.0]|27750.163531558544|
|Kentang Goreng|Yogyakarta|Sore       |3  |20000       |[3.0,20000.0,5.0,1.0,3.0]|64216.16074371864 |
|Kentang Goreng|Semarang  |Siang      |5  |22000       |[5.0,22000.0,5.0,2.0,2.0]|108722.79259154423|
|Kentang Goreng|Yogyakarta|Pagi       |2  |20000       |[2.0,20000.0,5.0,1.0,0.0]|54483.205871792496|
|Roti Bakar    |Magelang  |Malam      |2  |30000       |[2.0,30000.0,4.0,0.0,1.0]|58157.94412015849 |
+--------------+----------+-----------+---+------------+-------------------------+------------------+
only 

**B. Train/Test Split & Pelatihan Model**

In [3]:
from pyspark.ml.regression import LinearRegression

train_t11, test_t11 = df_tugas_final.randomSplit([0.75, 0.25], seed=101)
print(f"Jumlah data training (75%): {train_t11.count()}")
print(f"Jumlah data testing  (25%): {test_t11.count()}")

lr_t11 = LinearRegression(featuresCol="features", labelCol="total_penjualan")
model_t11 = lr_t11.fit(train_t11)

print("Model Regresi Kopi Nusantara berhasil dilatih!")
print("Koefisien fitur:", model_t11.coefficients)
print("Intercept:", model_t11.intercept)

Jumlah data training (75%): 1839
Jumlah data testing  (25%): 661
Model Regresi Kopi Nusantara berhasil dilatih!
Koefisien fitur: [22046.663271025758,3.022920868160672,-123.21425598657399,279.28452461677676,339.6387953231708]
Intercept: -66903.17766485889


**C. Evaluasi Model**

In [4]:
from pyspark.ml.evaluation import RegressionEvaluator

pred_t11 = model_t11.transform(test_t11)

rmse_t11 = RegressionEvaluator(labelCol="total_penjualan", predictionCol="prediction", metricName="rmse").evaluate(pred_t11)
r2_t11 = RegressionEvaluator(labelCol="total_penjualan", predictionCol="prediction", metricName="r2").evaluate(pred_t11)

print(f"RMSE Model Utama : Rp {rmse_t11:,.0f}")
print(f"R² Model Utama   : {r2_t11:.4f} ({r2_t11*100:.1f}%)")

RMSE Model Utama : Rp 10,901
R² Model Utama   : 0.9107 (91.1%)


**Interpretasi Evaluasi (Part C):**
Model regresi ini tergolong sangat baik dan layak digunakan untuk memprediksi omzet harian karena nilai R² berada di atas 0.90 (menjelaskan lebih dari 90% variasi data penjualan) dengan rata-rata kesalahan prediksi (RMSE) yang relatif kecil dibandingkan nilai transaksi. Hal ini menunjukkan bahwa kombinasi fitur yang digunakan sudah mampu menangkap pola pendapatan pesanan di Kopi Nusantara secara akurat.

**D. Eksperimen Perbandingan Fitur**

In [5]:
# Model kedua hanya menggunakan qty dan harga_satuan
assembler_dasar = VectorAssembler(
    inputCols=["qty", "harga_satuan"],
    outputCol="features_dasar"
)
df_tugas_dasar = assembler_dasar.transform(df_tugas_final)
train_dasar, test_dasar = df_tugas_dasar.randomSplit([0.75, 0.25], seed=101)

lr_dasar = LinearRegression(featuresCol="features_dasar", labelCol="total_penjualan")
model_dasar = lr_dasar.fit(train_dasar)
pred_dasar = model_dasar.transform(test_dasar)

r2_dasar = RegressionEvaluator(labelCol="total_penjualan", predictionCol="prediction", metricName="r2").evaluate(pred_dasar)

print(f"R² Model Utama (5 fitur) : {r2_t11:.4f}")
print(f"R² Model Kedua (2 fitur) : {r2_dasar:.4f}")
print(f"Selisih R²               : {abs(r2_t11 - r2_dasar):.4f}")

R² Model Utama (5 fitur) : 0.9107
R² Model Kedua (2 fitur) : 0.9108
Selisih R²               : 0.0001


**Kesimpulan Eksperimen Perbandingan Fitur (Part D):**
Penambahan informasi nama_menu, cabang, dan waktu_pesan tidak meningkatkan akurasi prediksi secara berarti, karena nilai R² pada model kedua (yang hanya memakai qty dan harga_satuan) menghasilkan skor yang nyaris identik dengan model utama. Secara logis, nilai total_penjualan memang ditentukan langsung oleh perkalian jumlah unit (qty) dan harga_satuan, sehingga atribut kategori seperti lokasi cabang maupun waktu pemesanan tidak memberikan pengaruh tambahan terhadap total tagihan pesanan.

**E. Prediksi Skenario Baru**

In [6]:
# Membuat 3 skenario pesanan dummy baru
data_dummy = spark.createDataFrame([
    ("Kopi Susu", "Magelang", "Pagi", 3, 20000),
    ("Americano", "Yogyakarta", "Siang", 2, 18000),
    ("Roti Bakar", "Semarang", "Malam", 5, 25000)
], ["nama_menu", "cabang", "waktu_pesan", "qty", "harga_satuan"])

# Transform menggunakan model StringIndexer dan VectorAssembler yang SAMA dari Bagian A
dummy_idx = indexer_menu.transform(data_dummy)
dummy_idx = indexer_cabang.transform(dummy_idx)
dummy_idx = indexer_waktu.transform(dummy_idx)
dummy_final = assembler_tugas.transform(dummy_idx)

# Prediksi total_penjualan menggunakan model utama
hasil_prediksi_dummy = model_t11.transform(dummy_final)

print("Hasil Prediksi Total Penjualan untuk 3 Skenario Baru:")
hasil_prediksi_dummy.select("nama_menu", "cabang", "waktu_pesan", "qty", "harga_satuan", "prediction").show()

spark.stop()
print("SparkSession Tugas 11 ditutup.")

Hasil Prediksi Total Penjualan untuk 3 Skenario Baru:


+----------+----------+-----------+---+------------+------------------+
| nama_menu|    cabang|waktu_pesan|qty|harga_satuan|        prediction|
+----------+----------+-----------+---+------------+------------------+
| Kopi Susu|  Magelang|       Pagi|  3|       20000| 59572.01525544525|
| Americano|Yogyakarta|      Siang|  2|       18000|32561.286619347826|
|Roti Bakar|  Semarang|      Malam|  5|       25000|119308.51121489711|
+----------+----------+-----------+---+------------+------------------+

SparkSession Tugas 11 ditutup.
